# 01 · Dimensions: bronze → silver → gold
All dimensions are built here. To add a dimension: add its source to the **Silver** section, then one `load_dim(...)` line to the **Gold** section.

| Gold dimension | Built from |
|---|---|
| `dim_customer` | `customers` |
| `dim_product` | `products` |
| `dim_customer_360` | five sources (CRM, billing, support, web, marketing) matched on e-mail |

Run order: this notebook first, then `02_facts` (facts look up these dimensions).

In [ ]:
%pip install databricks-labs-dqx -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
%run ./00_config

In [ ]:
LANDING = LANDING_DIMS      # dimension source files are read from this Volume folder

## Silver
One `load_silver` per source: **the columns and their types** (`type` or `type:transform`) and the business key. Rules come from `dq_rules.yml`.

In [ ]:
# ── sales domain ──
load_silver("customers", {
    "customer_id": "bigint", "name": "string", "email": "string:lower",
    "segment": "string:lower", "country": "string:upper", "updated_at": "timestamp"}, key="customer_id")

load_silver("products", {
    "product_id": "bigint", "product_name": "string", "category": "string:lower",
    "list_price": "decimal(12,2)", "updated_at": "timestamp"}, key="product_id")

In [ ]:
# ── customer feeds (one source system each) ──
load_silver("crm_accounts", {
    "account_id": "bigint", "full_name": "string", "email": "string:lower", "phone": "string:digits",
    "segment": "string:lower", "country": "string:upper", "owner": "string:lower", "updated_at": "timestamp"}, key="account_id")

load_silver("billing_customers", {
    "billing_id": "bigint", "email": "string:lower", "billing_name": "string", "plan": "string:lower",
    "status": "string:lower", "mrr": "decimal(10,2)", "country": "string:upper", "updated_at": "timestamp"}, key="billing_id")

load_silver("support_contacts", {
    "contact_id": "bigint", "email": "string:lower", "name": "string", "phone": "string:digits",
    "tier": "string:lower", "updated_at": "timestamp"}, key="contact_id")

load_silver("web_visitors", {
    "visitor_id": "bigint", "email": "string:lower", "first_seen": "timestamp", "last_seen": "timestamp",
    "visits": "int", "updated_at": "timestamp"}, key="visitor_id")

load_silver("marketing_leads", {
    "lead_id": "bigint", "email": "string:lower", "name": "string", "lead_source": "string:lower",
    "lead_status": "string:lower", "marketing_consent": "string:upper", "updated_at": "timestamp"}, key="lead_id")

### Combine the five feeds: one row per person
A person is matched **by e-mail**. For each attribute the first source that has a value wins (the order in each `coalesce` is the precedence, for example the support phone beats the CRM phone). The result is `silver.customer_unified`.

In [ ]:
def latest_by_email(name, pk):
    """The newest record per e-mail from one source."""
    w = Window.partitionBy("email").orderBy(F.desc("updated_at"), F.desc(pk))
    return spark.table(f"{SILVER}.{name}").withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

for view, (name, pk) in {"crm": ("crm_accounts", "account_id"), "billing": ("billing_customers", "billing_id"),
                         "support": ("support_contacts", "contact_id"), "web": ("web_visitors", "visitor_id"),
                         "mkt": ("marketing_leads", "lead_id")}.items():
    latest_by_email(name, pk).createOrReplaceTempView(view)

spark.sql("""
    WITH people AS (SELECT email FROM crm UNION SELECT email FROM billing UNION SELECT email FROM support
                    UNION SELECT email FROM web UNION SELECT email FROM mkt)
    SELECT p.email AS customer_key, p.email,
           coalesce(crm.full_name, billing.billing_name, support.name, mkt.name) AS full_name,
           coalesce(support.phone, crm.phone)                                    AS phone,
           crm.segment                                                           AS segment,
           coalesce(crm.country, billing.country)                                AS country,
           billing.plan, billing.status AS billing_status, billing.mrr,
           support.tier AS support_tier,
           web.first_seen, web.last_seen, web.visits,
           mkt.lead_source, mkt.lead_status, mkt.marketing_consent,
           crm.account_id AS crm_account_id, billing.billing_id, support.contact_id AS support_contact_id,
           web.visitor_id AS web_visitor_id, mkt.lead_id AS marketing_lead_id,
           crm.email     IS NOT NULL AS in_crm,     billing.email IS NOT NULL AS in_billing,
           support.email IS NOT NULL AS in_support, web.email     IS NOT NULL AS in_web,
           mkt.email     IS NOT NULL AS in_marketing,
           int(crm.email IS NOT NULL) + int(billing.email IS NOT NULL) + int(support.email IS NOT NULL)
             + int(web.email IS NOT NULL) + int(mkt.email IS NOT NULL) AS source_count
    FROM people p
    LEFT JOIN crm     ON crm.email     = p.email
    LEFT JOIN billing ON billing.email = p.email
    LEFT JOIN support ON support.email = p.email
    LEFT JOIN web     ON web.email     = p.email
    LEFT JOIN mkt     ON mkt.email     = p.email
""").write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.customer_unified")
print("silver.customer_unified", spark.table(f"{SILVER}.customer_unified").count(), "people")

## Gold
`load_dim(name, source, key, sk, attrs)`: the Silver table, its business key, the name of the surrogate key and the columns to carry over. `derived` adds columns calculated with a SQL expression.

`dim_customer_360` is a **Type 2** dimension (`load_dim_scd2`): `track` lists the columns whose change creates a new version (segment, country, plan, billing status, support tier, lifecycle). The other columns, like visits or last seen, are just kept up to date in the current row.

In [ ]:
load_dim("dim_customer", source="customers", key="customer_id", sk="customer_sk",
         attrs=["name", "email", "segment", "country"])

load_dim("dim_product", source="products", key="product_id", sk="product_sk",
         attrs=["product_name", "category", "list_price"])

load_dim_scd2("dim_customer_360", source="customer_unified", key="customer_key", sk="customer_sk",
         attrs=["email", "full_name", "phone", "segment", "country", "plan", "billing_status", "mrr", "support_tier",
                "first_seen", "last_seen", "visits", "lead_source", "lead_status", "marketing_consent",
                "crm_account_id", "in_crm", "in_billing", "in_support", "in_web", "in_marketing", "source_count"],
         derived={"lifecycle": """CASE WHEN in_billing AND billing_status IN ('active', 'past_due') THEN 'customer'
                                       WHEN in_billing THEN 'former customer'
                                       WHEN in_crm THEN 'prospect'
                                       WHEN in_marketing THEN 'lead'
                                       ELSE 'visitor or contact' END"""},
         track=["segment", "country", "plan", "billing_status", "support_tier", "lifecycle"])   # a change here = a new version

## Check the result

In [ ]:
display(dq_report().filter("source IN ('customers', 'products')"))
display(spark.sql(f"SELECT lifecycle, count(*) AS people FROM {GOLD}.dim_customer_360 WHERE is_current AND customer_sk <> -1 GROUP BY lifecycle ORDER BY people DESC"))
# people with more than one version: what changed and when
display(spark.sql(f"""SELECT customer_key, customer_sk, lifecycle, plan, billing_status, segment, effective_from, effective_to, is_current
                      FROM {GOLD}.dim_customer_360
                      WHERE customer_key IN (SELECT customer_key FROM {GOLD}.dim_customer_360 GROUP BY customer_key HAVING count(*) > 1)
                      ORDER BY customer_key, effective_from"""))
display(spark.sql(f"SELECT * FROM {GOLD}.dim_customer_360 WHERE source_count = 5 AND is_current LIMIT 3"))